# Can a glioblastoma change without growing? Cell states, macrophage cues and the limits of volume

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-assisted-glioma-research/blob/main/notebooks/05_tumor_microenvironment.ipynb)

*In Google Colab, the first code cell clones the public repository and puts its package on the path; Colab's own NumPy, Matplotlib and ipywidgets are used, not the locked versions.*
**Question:** Can a local microenvironment signal change which program tumour cells are in, without
changing how many tumour cells there are? And if so, what could an imaging measurement of tumour
size tell us about it?
**Evidence:** A synthetic two-state construction in days on a constructed millimetre plane (Figures
1–2 and two interactive explorers). It reproduces no published experiment and fits no patient data.

**Why this matters.** Glioblastoma is not one kind of cell. Single-cell studies show that its malignant
cells occupy several transcriptional programs and can move between them, and that signals from
neighbouring non-malignant cells, such as macrophages, can push them toward a mesenchymal-like (MES-like) program.
Most clinical imaging, by contrast, measures size: the volume of contrast-enhancing tumour or of
T2/FLAIR abnormality (signal change on T2-weighted or fluid-attenuated inversion recovery MRI).
If the *composition* of a tumour can change while its *size* stays the same, a
size measurement is blind to that change. This notebook builds the smallest model in which that
happens and asks exactly what such a model can and cannot tell us.

**Aims.** After working through this notebook you should be able to
1. explain, with references, what is meant by glioblastoma cell states, the MES-like program and a
   microenvironmental cue;
2. write down a two-state transition model and say what each term and unit means;
3. solve it by hand and read off its two key numbers, the equilibrium fraction and the time constant;
4. explain why a single late measurement identifies only the ratio of the rates, not the rates
   themselves;
5. say what a tumour volume can and cannot reveal about composition, and which measurements would be
   needed instead.

**Two tracks.** Most sections have an **oncological and biological track**, which connects the model to
glioma biology and clinical observation, and a **modeling track**, which gives the equations,
derivations and checks. Read either or both; the code cells and figures are shared. Acronyms and terms
from both fields are explained in the glossary after the next section, and all references are listed,
with links, at the end.

Other tumour states are aggregated into P; M is a MES-like program analogue. M is not the clinical
label "mesenchymal glioblastoma" and not a measured marker.

**Place in the perspective:** this cell-state composition example is distinct from non-malignant
astrocyte physiology (notebook 07) and from the growth–signal feedback of notebook 06.

## AI contribution and verification

**Question.** Glioblastoma cells are not one fixed type. Single-cell studies describe several
malignant cell programs and transitions between them, and some of those transitions appear to be
promoted by signals from non-malignant neighbours such as macrophages. That raises a question that
matters for imaging: if the *composition* of a tumour changes while its *size* does not, a size
measurement cannot see it. This notebook asks, in a model where every answer is exact:
- *Mechanism:* how much and how fast does a fixed local cue shift the balance between two programs?
- *Separation:* can composition change while total abundance stays fixed everywhere?
- *Inference:* which features of the result identify the rates, and which do not?

**Oncological and biological track.** Think of the three questions as: does a macrophage-rich region
change what the tumour cells there are doing; can that happen without the tumour growing or shrinking;
and what would we have to measure, and when, to estimate how quickly it happens?

**Modeling track.** The model is a linear system of two ordinary differential equations with a
conservation law. The three questions become: the equilibrium and time constant as functions of the
parameters; the conservation of $P+M$ at every grid point; and the structural identifiability of
$k_f$ and $k_r$ from different kinds of observation.

**AI contribution.**
- *Trained imaging AI:* none in this notebook.
- *Generative research assistance* (development only; records in `docs/ai-use/`): Codex desktop
  (6 October 2026; model snapshot not captured) helped implement the fixed-cue model, controlled
  comparisons, checks and shared figures
  ([record](../docs/ai-use/2026-10-06-biology-modeling.md)). On 10 October 2026 a Cursor agent (model
  and settings not captured) wrote an expanded version: the closed-form landmarks and both explorers
  in `glioma_ai.biology_interactive`, their tests and the explanatory text. On 13 October 2026 a Cursor
  agent (model and settings not captured) restructured the text into two tracks, added the glossary
  and the additional references, on request; every added reference was checked against Crossref and
  its PubMed abstract.
- *Conventional code* (rerun every time, no model): the exact fixed-cue solution and the checks.
  Rebuilding the notebook calls no language model.

**Design choices awaiting scientific review.** The project asked for a biological modeling example on
the tumour microenvironment, presented didactically with interactive demonstrations. Within that scope,
the following are assistant choices, with their reasons:
- *Two states instead of four or more:* the smallest model in which a program can shift; Neftel et
  al. describe four malignant states.
- *Reversible transitions:* a reverse rate gives a finite equilibrium and a single time constant;
  irreversible conversion would make every cued location drift to M eventually.
- *A fixed cue:* makes the equations linear and exactly solvable; real niches change over time.
- *k_forward = 0.08/day, k_reverse = 0.02/day:* time constants of 10 days (full cue) to 50 days (no
  cue), so the composition changes visibly within weeks. Not measured rates.
- *Fixed total abundance:* isolates composition from growth by construction, so that any change seen
  is a change of program, not of size.
- *The selection of references and the glossary entries.*

**Verification.**
- *Against an independent solution:* the package's spatial solver equals the closed form
  $m(t)=m^*+(m_0-m^*)e^{-(k_f s+k_r)t}$ at every location (Table 1).
- *Conservation:* P + M is unchanged at every grid point to about 1e-16.
- *Limits:* without a cue, M decays as $e^{-k_r t}$; the long-run fraction equals $k_f s/(k_f s+k_r)$.
- *Identifiability:* multiplying both rates by the same factor leaves the equilibrium unchanged but
  changes the time course (Figure 2, tested).
- *Not verified here:* any rate, cue amplitude or cue geometry; agreement with single-cell or spatial
  data; any MRI appearance.

**Evidence status.**
- *Figures 1–2 and the explorers:* synthetic constructions in stated units.
- *Development records:* actual contributions, summarized retrospectively and pending human
  scientific review. Comparative benefit of the assistance is unmeasured.

These roles describe development and provenance. [Assistance records](../docs/ai-workflow.md) retain
the review status; record completeness is separate from scientific quality.

## Glossary: acronyms and key terms

Refer back to these tables as needed. Terms are grouped by field; within each group they are roughly
in the order in which they appear.

**Oncological, biological and imaging terms**

| Term | Meaning |
| --- | --- |
| GBM | Glioblastoma. In the 2021 WHO classification ([Louis et al. 2021](https://doi.org/10.1093/neuonc/noab106)) the name is reserved for IDH-wildtype diffuse astrocytic tumours of grade 4 in adults. |
| IDH | Isocitrate dehydrogenase. Mutations in *IDH1* or *IDH2* define a separate group of diffuse gliomas with a different biology and prognosis; "IDH-wildtype" means no such mutation. |
| WHO CNS5 | The fifth edition (2021) of the World Health Organization classification of tumours of the central nervous system, which combines histology with molecular markers. |
| TME | Tumour microenvironment: the non-malignant cells (immune cells, blood vessels, astrocytes, neurons), the extracellular matrix and the chemical conditions (for example oxygen) surrounding the tumour cells. |
| TAM | Tumour-associated macrophages and microglia, the most abundant non-malignant cells in gliomas ([Hambardzumyan et al. 2016](https://doi.org/10.1038/nn.4185)). They come either from blood monocytes (Mo-TAM) or from brain-resident microglia. |
| Microglia | The resident immune cells of the brain. |
| Niche | A local region of the tumour with a characteristic environment, for example perivascular (around vessels), hypoxic, or peri-necrotic (around dead tissue). |
| Hypoxia | Low oxygen, typical near necrosis in glioblastoma. |
| scRNA-seq | Single-cell RNA sequencing: measures which genes are expressed in thousands of individual cells, so that cell types and states can be distinguished. |
| Spatial transcriptomics / proteomics | Methods that measure gene expression or proteins while keeping the position in the tissue section, so that states can be mapped. |
| Cell state, program | A recurring pattern of gene expression shared by many cells. A state is not a fixed cell type: cells can move between states. |
| MES-like, AC-like, OPC-like, NPC-like | The four main malignant cell states of [Neftel et al. 2019](https://doi.org/10.1016/j.cell.2019.06.024), resembling mesenchymal cells, astrocytes, oligodendrocyte-progenitor cells and neural-progenitor cells. |
| Plasticity | The ability of cells to change state. |
| Mesenchymal subtype | A *bulk-tumour* expression class ([Verhaak et al. 2010](https://doi.org/10.1016/j.ccr.2009.12.020); [Wang et al. 2017](https://doi.org/10.1016/j.ccell.2017.06.003)). It describes a whole sample and is not the same as the single-cell MES-like state. |
| TCGA | The Cancer Genome Atlas, a large public collection of tumour genomic data used in several of the cited studies. |
| OSM, OSMR, LIFR, GP130, STAT3 | Oncostatin M (a cytokine released by macrophages), its receptors OSMR or LIFR acting with the co-receptor GP130, and the transcription factor STAT3 activated downstream: the pathway by which [Hara et al. 2021](https://doi.org/10.1016/j.ccell.2021.05.002) found macrophages to induce MES-like states. |
| Ligand–receptor pair | A signalling molecule and the receptor it binds. Expression of both in neighbouring cells suggests, but does not prove, signalling. |
| Contrast enhancement | Brightening on T1-weighted MRI after a gadolinium contrast agent, where the blood–brain barrier is leaky. |
| T2/FLAIR abnormality | Signal change on T2-weighted or fluid-attenuated inversion recovery MRI, reflecting oedema and infiltrating tumour. |
| BBB | Blood–brain barrier: the tight lining of brain vessels that normally keeps contrast agent and many drugs out of the brain. |

**Modeling terms**

| Term | Meaning |
| --- | --- |
| ODE | Ordinary differential equation: an equation for how a quantity changes in time, here $dM/dt$. |
| State variable | A quantity the model tracks over time: here $P$ and $M$ at each location. |
| Rate constant | A number with unit 1/day giving the fraction of a population that changes state per day: $k_f$ and $k_r$. |
| First-order (linear) kinetics | The flow out of a state is proportional to how much is in it, as in radioactive decay or the elimination of most drugs. |
| Two-state (Markov) model | Each cell switches between two states at fixed rates, independently of its history; [Gupta et al. 2011](https://doi.org/10.1016/j.cell.2011.07.026) used such a model for cancer-cell state proportions. |
| Conservation law | A quantity that cannot change: here the total $T=P+M$ at each location, because cells only change state. |
| Closed-form solution | An exact formula for the solution, which can be used to check numerical code. |
| Equilibrium (steady state) $m^*$ | The fraction at which as many cells enter M as leave it, so the composition no longer changes. |
| Time constant $\tau$ | The time to cover about 63% of the remaining distance to equilibrium; after $3\tau$ about 95%. The half-life is $\tau\ln 2\approx 0.69\,\tau$. |
| Well-mixed | One location, with no spatial structure. |
| Spatial construction | A grid of locations, here 61 × 61 at 1 mm, each following the same equation with its own cue and total. Here there is no transport between locations. |
| Gaussian | A bell-shaped profile; here used for the total abundance and the cue, with width $\sigma$ in mm. |
| Normalized, dimensionless | Scaled to lie between 0 and 1, with no physical unit. |
| Identifiability | Whether parameters can be determined from a given kind of data. *Structural*: impossible even with perfect data (for example only the ratio $k_f/k_r$ from an equilibrium snapshot). *Practical*: possible in principle but poorly determined by limited, noisy data ([Raue et al. 2009](https://doi.org/10.1093/bioinformatics/btp358)). |
| Forward model | A model that predicts what an instrument would measure, for example MRI intensity from tissue properties. None is included here. |
| Synthetic construction | Inputs and parameters prescribed by the modeler, not measured. |

## 1. Biological background

**Oncological and biological track.**
- *Glioblastoma and its classification.* Under the 2021 WHO classification
  ([Louis et al. 2021](https://doi.org/10.1093/neuonc/noab106)), glioblastoma denotes IDH-wildtype diffuse astrocytic tumours of grade 4.
  The studies below are mostly of IDH-wildtype glioblastoma, and their findings should not be
  generalized to every glioma without evidence.
- *Bulk subtypes.* Expression profiling of whole tumour samples in TCGA defined proneural, neural,
  classical and mesenchymal subtypes, with *NF1* alterations characterizing the mesenchymal subtype
  ([Verhaak et al. 2010](https://doi.org/10.1016/j.ccr.2009.12.020)). A later analysis kept three tumour-intrinsic subtypes (proneural,
  classical, mesenchymal), related the mesenchymal subtype and *NF1* deficiency to more
  tumour-associated macrophages and microglia, and found that the subtype was retained at recurrence in
  55% of cases ([Wang et al. 2017](https://doi.org/10.1016/j.ccell.2017.06.003)). Composition can therefore change over the course of the
  disease.
- *Heterogeneity within one tumour.* Single-cell RNA sequencing showed that established subtype
  classifiers are expressed variably across individual cells within the same tumour
  ([Patel et al. 2014](https://doi.org/10.1126/science.1254257)). [Neftel et al. (2019)](https://doi.org/10.1016/j.cell.2019.06.024) described four main malignant states
  (MES-like, AC-like, OPC-like and NPC-like) in 28 tumours, showed with lineage tracing that cells can
  move between them, and found that their frequencies are influenced by genetic drivers (*CDK4*,
  *EGFR*, *PDGFRA* amplification and *NF1* mutation) and by the microenvironment.
- *The microenvironment as a driver.* Macrophages and microglia make up most of the non-malignant cells
  in gliomas ([Hambardzumyan et al. 2016](https://doi.org/10.1038/nn.4185); review of the brain TME:
  [Quail & Joyce 2017](https://doi.org/10.1016/j.ccell.2017.02.009)). [Hara et al. (2021)](https://doi.org/10.1016/j.ccell.2021.05.002) showed, in vitro and in vivo, that
  macrophages induce a transition of glioblastoma cells into MES-like states through macrophage-derived
  oncostatin M acting on OSMR or LIFR with GP130 and activating STAT3 (see also the
  [2025 figure correction](https://doi.org/10.1016/j.ccell.2025.05.007)).
- *Niches and spatial organization.* A monocyte-derived macrophage subset in the peri-necrotic niche
  acquires a hypoxia signature and makes tumour vessels more permeable
  ([Wang et al. 2024](https://doi.org/10.1016/j.ccell.2024.03.013)). Spatial multi-omics suggest that glioblastoma is organized by spatial
  segregation of states that adapt to inflammatory and metabolic stimuli
  ([Ravi et al. 2022](https://doi.org/10.1016/j.ccell.2022.05.009)), and that states form small local environments arranged in layers,
  with hypoxia as a long-range organizer ([Greenwald et al. 2024](https://doi.org/10.1016/j.cell.2024.03.029)).
- *What the model abstracts.* Together these findings suggest a *local cue* that shifts the
  composition of malignant states where it is present. The model keeps exactly that and nothing else;
  every rate is prescribed, not taken from these studies.

**Modeling track.** Describing cells that switch between states at fixed rates is a standard
abstraction: [Gupta et al. (2011)](https://doi.org/10.1016/j.cell.2011.07.026) showed that breast-cancer cell subpopulations sorted into one
state return toward the same equilibrium proportions, as a Markov model of stochastic state transitions
predicts. Here two states suffice. Because the transition rates are constant in time and the cue is
fixed, the model is linear and can be solved exactly, which makes it a good test bed for checking code
and for asking what different kinds of data can identify. Omitted on purpose: growth, death, migration,
immune-cell dynamics and any imaging forward model.

## 2. The model

$$\frac{dM}{dt}=k_f\,s\,P-k_r\,M,\qquad \frac{dP}{dt}=-\frac{dM}{dt}.$$

| Symbol | Meaning | Unit |
| --- | --- | --- |
| $P$ | abundance in all other programs, normalized | dimensionless |
| $M$ | abundance in the MES-like program analogue, normalized | dimensionless |
| $s$ | local cue between 0 and 1 (not a macrophage count or assay) | dimensionless |
| $k_f$ | forward transition rate at full cue | 1/day |
| $k_r$ | reverse transition rate | 1/day |

Adding the two equations gives $d(P+M)/dt=0$: the total $T=P+M$ is fixed at every location. There is
no proliferation, migration, death, immune-cell dynamics or MRI forward model.

**Oncological and biological track.** Read the first equation as a ledger for the MES-like cells at one
location. Cells *enter* M from the other programs at a rate that grows with the cue (for example, the
local macrophage signal) and with how many cells are still in P. Cells *leave* M and return to the other
programs at a constant rate, without needing any cue. The second equation says that every cell gained by
M is lost by P: no cell is born or dies. The cue $s$ is a dimensionless stand-in for "how strongly this
location pushes cells toward MES-like", not a measured macrophage density.

**Modeling track.** This is a linear, autonomous system of two ODEs with constant coefficients at each
location, equivalent to a two-state continuous-time Markov chain with transition rates $k_f s$
(P → M) and $k_r$ (M → P). The conservation law reduces it to one equation for the fraction
$m=M/T$ (next section). Locations are independent: there is no transport term, so the spatial problem
is a set of uncoupled scalar problems, one per grid point.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def repository_root():
    candidates = (Path.cwd(), *Path.cwd().parents)
    return next((p for p in candidates if (p / "pyproject.toml").is_file()), None)


if IN_COLAB and repository_root() is None:
    checkout = Path("/content/AI-assisted-glioma-research")
    if not checkout.is_dir():
        url = "https://github.com/arvidl/AI-assisted-glioma-research"
        subprocess.run(["git", "clone", "--depth", "1", url, str(checkout)], check=True)
    os.chdir(checkout)
    sys.path.insert(0, str(Path.cwd() / "src"))
if repository_root() is None:
    raise RuntimeError("Open this notebook in a checkout of the repository or in Google Colab.")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

from glioma_ai.biology import niche_transition
from glioma_ai.biology_examples import niche_example
from glioma_ai.biology_interactive import (
    draw_well_mixed,
    niche_landmarks,
    well_mixed_fraction,
    well_mixed_scenario,
)
from glioma_ai.notebook_display import show_figure

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
output = ROOT / "outputs/notebooks/microenvironment"
summary = niche_example(output)
summary["parameters"]

## 3. Solving it by hand

Write $m=M/T$ for the MES-like fraction. Substituting $P=T-M$ gives one linear equation,

$$\frac{dm}{dt}=k_f s-(k_f s+k_r)\,m,$$

with the solution

$$m(t)=m^*+(m_0-m^*)\,e^{-t/\tau},\qquad m^*=\frac{k_f s}{k_f s+k_r},\qquad \tau=\frac{1}{k_f s+k_r}.$$

- **Equilibrium** $m^*$: the fraction where forward and reverse transitions balance. It depends on the
  rates only through their *ratio* $k_f/k_r$.
- **Time constant** $\tau$: how fast the equilibrium is approached. It depends on the rates' *sum*.
- **No cue** ($s=0$): $m^*=0$ and M decays as $e^{-k_r t}$.

**Oncological and biological track.** Two numbers summarize everything. The *equilibrium* is the
composition the location settles into: with the chosen rates, 80% MES-like at full cue, 67% at half
cue, 50% at a quarter cue and none without a cue. The *time constant* says how quickly it gets there:
10 days at full cue, 50 days without a cue. As with a drug half-life, about 63% of the remaining change
happens in one time constant and 95% in three. A stronger cue therefore both raises the final
MES-like share and makes the change happen sooner.

**Modeling track.** With $P=T-M$ and $T$ constant, $dM/dt=k_f s(T-M)-k_r M$; dividing by $T$ gives
the equation for $m$. It is a first-order linear ODE, $\dot m=a-bm$ with $a=k_f s$ and $b=k_f s+k_r$,
solved with the integrating factor $e^{bt}$. Two structural facts follow: $m^*=a/b$ is invariant when
both rates are multiplied by the same factor, while $\tau=1/b$ is not. The package solver works on the
whole plane; the next cell checks it against these formulas to a relative tolerance of 1e-12.

In [ ]:
rows = []
for cue in (0.0, 0.25, 0.5, 1.0):
    marks = niche_landmarks(0.08, 0.02, cue)
    days = np.array([10.0, 30.0])
    closed = well_mixed_fraction(0.08, 0.02, cue, days, 0.1)
    package = []
    for day in days:
        _, mes = niche_transition(
            np.full((3, 3), 0.63), np.full((3, 3), 0.07), np.full((3, 3), cue), 0.08, 0.02, day
        )
        package.append(mes[1, 1] / 0.7)
    np.testing.assert_allclose(package, closed, rtol=1e-12)
    rows.append(
        f"| {cue:.2f} | {marks['equilibrium_fraction']:.3f} | {marks['time_constant_days']:.1f} | "
        f"{closed[0]:.3f} | {package[0]:.3f} | {closed[1]:.3f} |"
    )
display(
    Markdown(
        "**Table 1. Closed form against the package solver (k_f = 0.08/day, k_r = 0.02/day, "
        "m0 = 0.1; synthetic).**\n\n"
        "| Cue s | Equilibrium m* | τ (days) | m(10 d), closed form | m(10 d), package | "
        "m(30 d), closed form |\n| --- | --- | --- | --- | --- | --- |\n" + "\n".join(rows)
    )
)

**How to read Table 1.** Each row is one cue strength. The second and third columns are the
equilibrium and time constant from the formulas above. The next two columns compare the closed form
with the package's spatial solver after 10 days, starting from 10% MES-like; they agree to 12 digits
(the cell asserts it). The last column shows the closed form after 30 days.

**Interpretation.** With no cue the fraction falls from 0.10 toward 0, with a 50-day time constant
(0.082 after 10 days, 0.055 after 30). At full cue it rises toward 0.80 with a 10-day time constant
(0.54 after 10 days, 0.77 after 30). Intermediate cues give intermediate equilibria *and* intermediate
speeds. A stronger cue therefore both raises the equilibrium and speeds up the approach.

## 4. The spatial construction

Now let the cue vary across a constructed 61 × 61 mm plane. The total abundance is a fixed Gaussian
(peak 0.7, width 9 mm) and the cue a Gaussian offset 7 mm to the right (width 8 mm). Every location
starts with 10% of its abundance in M and evolves independently by the equation above; there is no
transport between locations.

**Oncological and biological track.** Picture a tumour whose cell density is highest in its centre and
a macrophage-rich region sitting off-centre, overlapping part of the tumour. Cells inside the
macrophage-rich region shift toward MES-like; cells far from it do not. Because no cells are added or
removed, any density image would look the same before and after.

**Modeling track.** Each grid point is an independent copy of the well-mixed problem with its own
$s(x,y)$ and $T(x,y)$, so the closed form applies pointwise. The *overall* fraction is the
abundance-weighted mean $\sum_{x,y} M/\sum_{x,y} T$; it depends on where the cue overlaps the tumour,
not only on the cue itself. The next cell asserts conservation to 1e-15 and the no-cue decay limit.

In [ ]:
display(Image(filename=str(output / "microenvironment.png")))
arrays = np.load(output / "synthetic_niche.npz")
np.testing.assert_allclose(
    arrays["other"] + arrays["mes_like"], arrays["total"], rtol=0, atol=1e-15
)
assert summary["total_density_change_max_abs"] < 1e-15
assert summary["final_mes_like_fraction"] > summary["no_cue_final_mes_like_fraction"]

# Independent limit: without a forward cue, M decays at the prescribed reverse rate.
other = np.full((3, 3), 0.72)
mes = np.full((3, 3), 0.08)
_, no_cue = niche_transition(other, mes, np.zeros((3, 3)), 0.08, 0.02, 10)
np.testing.assert_allclose(no_cue, 0.08 * np.exp(-0.02 * 10))
{
    "initial_fraction": summary["initial_mes_like_fraction"],
    "cue_final_fraction": summary["final_mes_like_fraction"],
    "no_cue_final_fraction": summary["no_cue_final_mes_like_fraction"],
    "maximum_local_total_change": summary["total_density_change_max_abs"],
}

**Figure 1. Composition changes at fixed total density (synthetic construction).** From left: the
prescribed total abundance, unchanged throughout; the prescribed cue; the MES-like fraction after 10
days; and the overall MES-like fraction over 10 days with and without the cue. All maps share a colour
scale from 0 to 1.

**Reading the panels.**
- *Total density (first panel):* a centred Gaussian with peak 0.7. It is identical at day 0 and day 10;
  this is what a volume or density measurement would see.
- *Cue (second panel):* peak 1 at x = 7 mm, so it overlaps the right half of the tumour more than the
  left.
- *MES-like fraction at day 10 (third panel):* highest, about 0.54, at the centre of the cue, where the
  time constant is shortest (10 days). On a scale up to 1 it looks modest because the equilibrium of
  0.8 has not yet been reached. Far from the cue the fraction has fallen slightly, from 0.10 to about
  0.08.
- *Overall fraction (fourth panel):* rises from 0.10 to 0.28 in 10 days with the cue and falls to 0.08
  without it. With the cue left in place it would approach about 0.49, the abundance-weighted mean of
  the local equilibria.

**Interpretation.** A substantial change in composition, nearly a tripling of the overall MES-like
share in ten days, happens with no change at all in the density map. A clinician comparing two volume
measurements would see a stable tumour.

## 5. What the rates can and cannot tell us

Because the equilibrium depends only on $k_f/k_r$, doubling both rates gives the *same* long-run
composition, reached twice as fast. A single late measurement of composition therefore identifies the
ratio but not the rates; a time course is needed for the rates themselves.

**Oncological and biological track.** Suppose a biopsy taken long after a cue was established shows
80% MES-like cells. That is compatible with a slow transition (weeks) and with a fast one (days). To
know how fast the tumour can change, and so how often it would need to be sampled, one needs
measurements at two or more times, ideally early in the change.

**Modeling track.** This is *structural* non-identifiability: from equilibrium data alone, the
parameter pair $(k_f,k_r)$ is determined only up to a common factor, whatever the data quality. A time
course at the same cue identifies $\tau$, and with $m^*$ both rates. With noisy data, the question
becomes one of *practical* identifiability, which profile-likelihood methods address
([Raue et al. 2009](https://doi.org/10.1093/bioinformatics/btp358)).

In [ ]:
run = well_mixed_scenario(rate_scale=2.0)
figure = plt.figure(figsize=(12, 4.2))
draw_well_mixed(figure, run)
show_figure(figure)
assert run["landmarks"]["equilibrium_fraction"] == run["scaled_landmarks"]["equilibrium_fraction"]
print(
    f"Equilibrium {run['landmarks']['equilibrium_fraction']:.2f} for both; time constants "
    f"{run['landmarks']['time_constant_days']:.0f} and "
    f"{run['scaled_landmarks']['time_constant_days']:.0f} days."
)

**Figure 2. Same equilibrium, different speed (synthetic).** Left: the MES-like fraction at full cue
for the reference rates (solid) and for both rates doubled (dashed). Right: the equilibrium fraction as
a function of cue strength.

**Reading the panels.** On the left, both curves start at 0.10 and level off at the same dotted line,
0.80. The vertical lines mark the time constants, 10 days for the reference rates and 5 days for the
doubled rates: by then each curve has covered about 63% of the way. Measured only at day 60, the two
would be indistinguishable; measured at day 5 they differ clearly (about 0.38 against 0.54). On the
right, the equilibrium rises steeply for weak cues and then flattens toward 0.80 at full cue: doubling a
weak cue nearly doubles the equilibrium, while doubling a strong one adds little. This curve depends on
$k_f/k_r$ only, so it is the same for both rate settings on the left.

## 6. Explore it yourself

**Why explore.** The figures show one choice of rates and cue. The explorers let you change them and
see which features of the result move and which stay fixed, which is how one develops an intuition for
what a measurement could reveal. For each task, first *predict* what will happen, then move the
sliders, then *explain* what you see. Moving a slider needs a running kernel; in a static rendering only
the initial state is shown.

**Well-mixed explorer.** One location, exact solution. Sliders: the forward and reverse rates
$k_f$ and $k_r$ (1/day), the cue $s$, the initial fraction $m_0$, and a common factor applied to both
rates for the dashed curve. Things to try:
1. Set the cue to 0 and the initial fraction to 0.5. What is the half-life of M, and which rate sets
   it? (With $k_r$ = 0.02/day it is $\ln 2/k_r\approx$ 35 days.)
2. Change the common rate factor. Which curve features move, and which stay fixed?
3. Set $k_r$ to 0. What is the equilibrium now, for any cue above 0? Why is that a strong assumption?

In [ ]:
from glioma_ai.biology_interactive import well_mixed_explorer

well_mixed_explorer()

**Spatial explorer.** The construction of Figure 1, with a movable cue. Sliders: both rates, the cue
amplitude, the cue centre and width, and the displayed day (up to 60). The left panel shows the cue in
green with contours of total abundance, the middle panel the local MES-like fraction on the displayed
day, and the right panel the overall fraction over 60 days with its long-run limit. Things to try:
1. Move the cue centre from 7 mm to 20 mm. At day 10 the overall fraction falls from 0.28 to about
   0.15, and its long-run limit from 0.49 to 0.20, because the cue now sits where there is little
   tumour, although the cue itself is unchanged.
2. Narrow the cue to 2 mm. The local fraction at its centre still rises exactly as before (0.54 at day
   10, approaching 0.8), but the overall fraction barely moves (0.10 at day 10, against 0.08 without a
   cue). Local and overall measures can disagree.
3. Advance the displayed day to 60. Which locations have reached their local equilibrium? Near the cue
   centre (time constant 10 days) the fraction is close to 0.8; far from it, where the time constant
   approaches 50 days, it is still changing.

The right-hand title reports the largest change in total abundance anywhere on the plane; it stays at
rounding level whatever you do.

**Oncological and biological track.** Tasks 1 and 2 are about sampling: a biopsy from the cue centre
and a whole-tumour average can tell different stories, and where the macrophage-rich region sits
relative to the bulk of the tumour matters as much as how strong it is.

**Modeling track.** The long-run limit is $\sum T\,m^*(s)/\sum T$, a weighted average of a nonlinear
(saturating) function of $s$; it is not $m^*$ evaluated at the average cue. Task 3 illustrates that
with a spatially varying cue there is no single time constant for the whole tumour.

In [ ]:
from glioma_ai.biology_interactive import niche_explorer

niche_explorer()

## 7. Biological interpretation and the imaging connection

**Oncological and biological track.** The construction separates cell abundance from state
composition. A volume measurement alone does not supply the missing state measurements. We have not
simulated MRI intensity, contrast leakage, or a causal patient mechanism. Some of the processes
discussed here do have imaging-relevant consequences, for example the hypoxia-associated macrophages
that make vessels more permeable ([Wang et al. 2024](https://doi.org/10.1016/j.ccell.2024.03.013)), and vessel permeability is what contrast
enhancement reflects; but no link between MES-like composition and any MRI feature is modeled or
claimed here.

To test a biological hypothesis, one would need an appropriate molecular or protein state assay,
spatial information about the niche, and a perturbation or longitudinal design. A ligand-receptor
association alone is not proof that signalling caused a transition. IDH-wildtype GBM evidence should
not be silently generalized to every glioma subtype.

**Modeling track.** Each missing ingredient would change the mathematics. Growth or death would break
the conservation law; migration would couple neighbouring locations through a transport term; a cue
that responds to the tumour would make the system nonlinear (notebook 06 takes that step for
growth); and an imaging forward model would be needed before any comparison with MRI. Each addition
also adds parameters, which makes the identifiability questions of section 5 harder, not easier.

## 8. Summary and exercises

**What this notebook shows.**
- *Composition can change at fixed size.* A fixed local cue shifts the share of cells in a MES-like
  program while the total abundance stays unchanged everywhere, to rounding error (Figure 1).
- *Two numbers govern the change.* The equilibrium $k_f s/(k_f s+k_r)$ sets where the composition goes;
  the time constant $1/(k_f s+k_r)$ sets how fast (Table 1).
- *Geometry matters.* The overall effect depends on where the cue overlaps the tumour, not only on its
  strength; local and overall measures can disagree (spatial explorer).
- *A snapshot is not enough.* A late measurement identifies only the rate ratio; the rates themselves
  need a time course (Figure 2).

**What it does not show.**
- *That macrophages cause this shift in patients.* The cited studies motivate the cue; the model does
  not test or reproduce them.
- *Any real rate, cue strength or cue geometry.* All are prescribed for teaching.
- *Anything visible on MRI.* No imaging forward model is included, and no MRI feature of MES-like
  composition is claimed.
- *Growth, treatment effects or outcome.* Total abundance is held fixed by construction.

**Exercises: oncological and biological track.**
1. *Same size, different composition.* The next cell compares three cue strengths at the same total and
   rates after 10 days. What changes, and what stays fixed? Which measurement (for example a biopsy with
   a state assay, or spatial transcriptomics) could distinguish these three tumours, and which could
   not (for example a volume measurement)?
2. *Reversible or not?* Remove the reverse transition in the well-mixed explorer ($k_r$ = 0). Explain
   what this assumption means biologically, and which kind of experiment could tell reversible from
   irreversible transitions (compare the lineage tracing of Neftel et al. and the sorted
   subpopulations of Gupta et al.).
3. *A different cue.* Greenwald et al. (2024) identify hypoxia as a long-range organizer of cell
   states, and Wang et al. (2024) describe macrophages reprogrammed in the hypoxic peri-necrotic niche.
   How would you redraw the cue map if hypoxia, rather than a macrophage cluster, were the
   cue? Which feature of routine MRI might serve as a rough proxy for its location, and why only as a
   proxy?
4. *Recurrence.* Wang et al. (2017) found that the bulk subtype was retained at recurrence in 55% of
   cases. In this model, what would have to happen between surgery and recurrence for the composition
   to change without any change in size, and what sampling would detect it?

**Exercises: modeling track.**
1. *Identify both rates.* Design a two-time-point measurement that would identify both rates in this
   model. How many locations and which cue levels would you need? What changes if each measurement
   has 10% noise?
2. *Long-run limit.* Show that the long-run overall fraction is $\sum T\,m^*(s)/\sum T$, and check the
   value 0.49 for the default spatial explorer. Why is it not equal to $m^*$ at the mean cue?
3. *Three states.* Extend the model to three states as a continuous-time Markov chain. Write the rate
   matrix, show that its columns sum to zero (conservation), and find the equilibrium as its null
   vector. Compare with the model of Gupta et al. (2011).
4. *Add transport.* Add diffusion of cells between neighbouring grid points. Which of the checks used
   here (closed form, conservation, no-cue limit) would still hold, and in what form?

In [ ]:
comparison = []
for strength in (0.0, 0.5, 1.0):
    p, q = niche_transition(other, mes, np.full((3, 3), strength), 0.08, 0.02, 10)
    comparison.append(
        {
            "cue": strength,
            "total": float((p + q).mean()),
            "mes_fraction": float(q.sum() / (p + q).sum()),
        }
    )
comparison

## Assistance and acceptance evidence

This implementation was developed with Codex and expanded on 10 and 13 October 2026 with a Cursor
agent. The limiting-case, closed-form and conservation checks are executable evidence about the code.
The development records are retrospective summaries, not invented dialogue or a completed human
biological review. See `docs/ai-use/2026-10-06-biology-modeling.md` and notebook 04.

## References

All links were checked on 13 October 2026; the summaries in this notebook follow the published
abstracts.

**Glioma biology and classification**
- Louis DN et al. The 2021 WHO Classification of Tumors of the Central Nervous System: a summary.
  *Neuro-Oncology* 2021. [doi:10.1093/neuonc/noab106](https://doi.org/10.1093/neuonc/noab106)
- Verhaak RGW et al. Integrated genomic analysis identifies clinically relevant subtypes of
  glioblastoma characterized by abnormalities in PDGFRA, IDH1, EGFR, and NF1. *Cancer Cell* 2010.
  [doi:10.1016/j.ccr.2009.12.020](https://doi.org/10.1016/j.ccr.2009.12.020)
- Wang Q et al. Tumor evolution of glioma-intrinsic gene expression subtypes associates with
  immunological changes in the microenvironment. *Cancer Cell* 2017.
  [doi:10.1016/j.ccell.2017.06.003](https://doi.org/10.1016/j.ccell.2017.06.003)
- Patel AP et al. Single-cell RNA-seq highlights intratumoral heterogeneity in primary glioblastoma.
  *Science* 2014. [doi:10.1126/science.1254257](https://doi.org/10.1126/science.1254257)
- Neftel C et al. An integrative model of cellular states, plasticity, and genetics for glioblastoma.
  *Cell* 2019. [doi:10.1016/j.cell.2019.06.024](https://doi.org/10.1016/j.cell.2019.06.024)

**Microenvironment, macrophages and niches**
- Hambardzumyan D, Gutmann DH, Kettenmann H. The role of microglia and macrophages in glioma
  maintenance and progression. *Nature Neuroscience* 2016. [doi:10.1038/nn.4185](https://doi.org/10.1038/nn.4185)
- Quail DF, Joyce JA. The microenvironmental landscape of brain tumors. *Cancer Cell* 2017.
  [doi:10.1016/j.ccell.2017.02.009](https://doi.org/10.1016/j.ccell.2017.02.009)
- Hara T et al. Interactions between cancer cells and immune cells drive transitions to
  mesenchymal-like states in glioblastoma. *Cancer Cell* 2021.
  [doi:10.1016/j.ccell.2021.05.002](https://doi.org/10.1016/j.ccell.2021.05.002); figure correction 2025:
  [doi:10.1016/j.ccell.2025.05.007](https://doi.org/10.1016/j.ccell.2025.05.007)
- Wang W et al. Identification of hypoxic macrophages in glioblastoma with therapeutic potential for
  vasculature normalization. *Cancer Cell* 2024. [doi:10.1016/j.ccell.2024.03.013](https://doi.org/10.1016/j.ccell.2024.03.013)
- Ravi VM et al. Spatially resolved multi-omics deciphers bidirectional tumor-host interdependence in
  glioblastoma. *Cancer Cell* 2022. [doi:10.1016/j.ccell.2022.05.009](https://doi.org/10.1016/j.ccell.2022.05.009)
- Greenwald AC et al. Integrative spatial analysis reveals a multi-layered organization of
  glioblastoma. *Cell* 2024. [doi:10.1016/j.cell.2024.03.029](https://doi.org/10.1016/j.cell.2024.03.029)

**Modeling**
- Gupta PB et al. Stochastic state transitions give rise to phenotypic equilibrium in populations of
  cancer cells. *Cell* 2011. [doi:10.1016/j.cell.2011.07.026](https://doi.org/10.1016/j.cell.2011.07.026)
- Raue A et al. Structural and practical identifiability analysis of partially observed dynamical
  models by exploiting the profile likelihood. *Bioinformatics* 2009.
  [doi:10.1093/bioinformatics/btp358](https://doi.org/10.1093/bioinformatics/btp358)

Continue with **06_neuron_glioma_feedback.ipynb**, the main talk example.